# 04b. Parallel Cell-Type Sorting via dsub

Submits all-7-cell-type LLR sorting jobs to Google Batch via `dsub`,  
enabling parallel processing of all samples simultaneously.

**Use this notebook when:**
- Sorting a large cohort (>50 samples) where sequential processing would take days
- Running the full production dataset (thousands of samples)

**For small-scale or testing use `04_digital_sorting.ipynb` instead.**

## Algorithm
Each dsub VM processes one PAT file:
1. Download marker BED files from GCS
2. Download PAT file
3. Run single-pass LLR scoring across all 7 cell types simultaneously
4. For each cell type: bgzip, index, pat2beta the sorted reads
5. Upload results to `$BUCKET/results/cell_sorted/{CellType}/`

## Modes
- `MODE = "TEST"` — 3 samples, stable VM, waits for completion  
- `MODE = "PRODUCTION"` — all samples, preemptible VM, submits asynchronously

**Estimated cost:** ~\$0.01–0.05/sample × ~25 min/sample

## Required GCS Resources
- `$BUCKET/resources/wgbstools_package.tar.gz`
- `$BUCKET/resources/hg38_cpg_refs.tar.gz` (CpG.chrome.size + chrom reference)
- `$BUCKET/resources/wgbs_wheels/` (pandas, numpy, scipy)
- `$BUCKET/resources/apt_debs/` (bgzip, tabix)
- Marker BEDs at `$BUCKET/results/markers/markers_S2/` (from notebook 03)

## Cell 1 — Imports and dsub Installation

In [ ]:
import subprocess, sys, os, time
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

PROJECT_ID = os.environ['GOOGLE_PROJECT']
BUCKET     = os.environ['WORKSPACE_BUCKET']
REGION     = 'us-central1'

print('Installing dsub...')
r = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', 'dsub', '-q'],
    capture_output=True, text=True
)
if r.returncode == 0:
    ver = subprocess.run('dsub --version', shell=True, capture_output=True, text=True)
    print(f'dsub ready — {ver.stdout.strip() or ver.stderr.strip()}')
else:
    print(f'dsub install failed: {r.stderr[-500:]}')

print(f'\nProject: {PROJECT_ID}')
print(f'Bucket : {BUCKET}')

## Cell 2 — Master Configuration

In [ ]:
# ════════════════════════════════════════════════════════════════
# Cell Type Sorting — all 7 blood cell types
# INPUT:  PAT files from any preprocessing batch
# OUTPUT: results/cell_sorted/{CellType}/{sample_key}_{CellType}.*
# ════════════════════════════════════════════════════════════════

MODE = 'TEST'    # ← 'TEST' (3 samples, stable VM) or 'PRODUCTION' (all, preemptible)

# ── Fixed GCS resource paths ──────────────────────────────────────
WGBS_TOOLS_TAR    = f'{BUCKET}/resources/wgbstools_package.tar.gz'
HG38_REFS_TAR     = f'{BUCKET}/resources/hg38_cpg_refs.tar.gz'
WHEELS_GCS        = f'{BUCKET}/resources/wgbs_wheels'
DEBS_GCS          = f'{BUCKET}/resources/apt_debs'
WORKER_PY_FILE    = 'celltype_worker.py'
WORKER_SCRIPT_GCS = f'{BUCKET}/resources/celltype_worker.py'
SCRIPT_FILE       = 'wgbs_celltype_job.sh'
CONFIG_FILE       = 'dsub_config.txt'
LOCAL_CSV         = 'celltype_manifest.csv'
TSV_FILE          = f'dsub_jobs_celltype_{MODE.lower()}.tsv'

# ── Marker BED GCS paths (from notebook 03) ──────────────────────
TARGET_BEDS = {
    'Myeloid':     f'{BUCKET}/results/markers/markers_S2/coarse_Myeloid_vs_Lymphoid/Markers.Myeloid.bed',
    'Lymphoid':    f'{BUCKET}/results/markers/markers_S2/coarse_Lymphoid_vs_Myeloid/Markers.Lymphoid.bed',
    'T_Cell':      f'{BUCKET}/results/markers/markers_S2/one_vs_all/T_Cell_vs_all/Markers.T_Cell.bed',
    'B_Cell':      f'{BUCKET}/results/markers/markers_S2/one_vs_all/B_Cell_vs_all/Markers.B_Cell.bed',
    'NK_Cell':     f'{BUCKET}/results/markers/markers_S2/one_vs_all/NK_Cell_vs_all/Markers.NK_Cell.bed',
    'Monocyte':    f'{BUCKET}/results/markers/markers_S2/one_vs_all/Monocyte_vs_all/Markers.Monocyte.bed',
    'Granulocyte': f'{BUCKET}/results/markers/markers_S2/one_vs_all/Granulocyte_vs_all/Markers.Granulocyte.bed',
}

# ── Input PAT folders (all preprocessing batches) ────────────────
IN_PAT_PREFIXES = [
    f'{BUCKET}/results/jhu_ont',
    f'{BUCKET}/results/bcm_ont',
    f'{BUCKET}/results/dsub_UW_REVIO',
    f'{BUCKET}/results/dsub_BROAD_REVIO',
    f'{BUCKET}/results/dsub_BROAD_SEQUEL',
    # Sequential batches from notebook 01
    f'{BUCKET}/results/bi_revio',
    f'{BUCKET}/results/bcm_revio',
    f'{BUCKET}/results/bi_sequel',
]

# ── Output GCS root ───────────────────────────────────────────────
OUT_ROOT_GCS = f'{BUCKET}/results/cell_sorted'

MODE_SETTINGS = {
    'TEST': {
        'n_samples':   3,
        'disk_gb':     30,
        'preemptible': False,
        'wait':        True,
    },
    'PRODUCTION': {
        'n_samples':   None,
        'disk_gb':     30,
        'preemptible': True,
        'wait':        False,
    },
}
cfg = MODE_SETTINGS[MODE]

print(f'{"━"*55}')
print(f'  MODE:        {MODE}')
print(f'  Cell types:  {len(TARGET_BEDS)}')
print(f'  Input dirs:  {len(IN_PAT_PREFIXES)}')
print(f'  Output:      {OUT_ROOT_GCS}/')
print(f'  Disk/VM:     {cfg["disk_gb"]}GB')
print(f'  Preemptible: {cfg["preemptible"]}')
print(f'  ~min/sample: 25 min')
print(f'{"━"*55}')

## Cell 3 — Stage hg38 CpG References Locally

Downloads wgbstools and the hg38 CpG reference files needed by the worker.

In [ ]:
HOME           = Path(os.getcwd())
REFS_DIR       = HOME / 'wgbs_tools' / 'references' / 'hg38'
TAR_LOCAL      = 'hg38_cpg_refs.tar.gz'
WGBS_TAR_LOCAL = 'wgbstools_package.tar.gz'

print('Checking wgbstools locally...')
if not (HOME / 'wgbs_tools').exists():
    print('  Downloading wgbstools...')
    subprocess.run(f'gsutil -q cp {WGBS_TOOLS_TAR} {WGBS_TAR_LOCAL}', shell=True, check=True)
    subprocess.run(f'tar -xzf {WGBS_TAR_LOCAL}', shell=True, check=True)
    print('  Done')
else:
    print('  Already present')

print('Checking hg38 CpG refs...')
if not REFS_DIR.exists():
    print('  Downloading hg38 CpG refs...')
    subprocess.run(f'gsutil -q cp {HG38_REFS_TAR} {TAR_LOCAL}', shell=True, check=True)
    subprocess.run(f'tar -xzf {TAR_LOCAL}', shell=True, check=True)
    print('  Done')
else:
    print('  Already present')

TOOL_BIN = HOME / 'wgbs_tools' / 'wgbstools'
CPG_REFS = REFS_DIR
print(f'\nwgbstools: {TOOL_BIN} (exists={TOOL_BIN.exists()})')
print(f'hg38 refs: {CPG_REFS} (exists={CPG_REFS.exists()})')

## Cell 4 — Discover VPC Network Settings

In [ ]:
print('Discovering VPC network settings...\n')

METADATA = 'http://metadata.google.internal/computeMetadata/v1'
HEADER   = '-H "Metadata-Flavor: Google"'

def curl_meta(path):
    r = subprocess.run(
        f'curl -sf {HEADER} "{METADATA}/{path}"',
        shell=True, capture_output=True, text=True
    )
    return r.stdout.strip() if r.returncode == 0 else None

PET_SA = subprocess.run(
    "gcloud config list account --format='value(core.account)'",
    shell=True, capture_output=True, text=True
).stdout.strip()
assert 'iam.gserviceaccount.com' in PET_SA, f'Unexpected identity: {PET_SA}'

network_uri    = curl_meta('instance/network-interfaces/0/network')
subnetwork_uri = curl_meta('instance/network-interfaces/0/subnetwork')

def short_net(uri):
    if uri and 'projects/' in uri:
        parts = uri.split('/')
        try:
            proj = parts[parts.index('projects') + 1]
            return f'projects/{proj}/{parts[-2]}/{parts[-1]}'
        except Exception:
            pass
    return uri or 'UNKNOWN'

NETWORK_ARG    = short_net(network_uri)
SUBNETWORK_ARG = short_net(subnetwork_uri)

print(f'Service account : {PET_SA}')
print(f'Network         : {NETWORK_ARG}')
print(f'Subnetwork      : {SUBNETWORK_ARG}')

with open(CONFIG_FILE, 'w') as f:
    f.write(f'PET_SA={PET_SA}\n')
    f.write(f'NETWORK={NETWORK_ARG}\n')
    f.write(f'SUBNETWORK={SUBNETWORK_ARG}\n')
print(f'Saved → {CONFIG_FILE}')

## Cell 5 — Build PAT File Manifest

In [ ]:
def gsutil_ls(prefix):
    try:
        out = subprocess.check_output(
            ['gsutil', 'ls', prefix], text=True, stderr=subprocess.DEVNULL
        )
        return [x.strip() for x in out.splitlines() if x.strip()]
    except subprocess.CalledProcessError:
        return []

def make_sample_key(pat_gcs):
    p = Path(pat_gcs)
    return f'{p.parent.name}__{p.name.replace(".pat.gz", "")}'

def is_done(sample_key, done_by_target):
    return all(sample_key in done_by_target.get(t, set()) for t in TARGET_BEDS)


# Collect all PAT files
print('Scanning PAT folders...\n')
all_pats = []
for prefix in IN_PAT_PREFIXES:
    found = gsutil_ls(f'{prefix}/*.pat.gz')
    name  = prefix.split('results/')[-1]
    print(f'  {len(found):>5}  {name}')
    all_pats.extend(found)

all_pats = sorted(set(all_pats))
print(f'\n  Total PAT files: {len(all_pats)}')

# Build manifest
print('\nBuilding manifest...')
records = []
for pat_gcs in all_pats:
    p = Path(pat_gcs)
    records.append({
        'sample_id':     p.name.replace('.pat.gz', ''),
        'source_folder': p.parent.name,
        'sample_key':    make_sample_key(pat_gcs),
        'pat_gcs':       pat_gcs,
    })

MANIFEST_DF = pd.DataFrame(records)

# Check which are already done (fast scan)
print('Checking completion status...')
done_by_target = {}
for target in TARGET_BEDS:
    r = subprocess.run(
        f'gsutil ls {OUT_ROOT_GCS}/{target}/*.beta 2>/dev/null',
        shell=True, capture_output=True, text=True
    )
    done_keys = set()
    for l in r.stdout.splitlines():
        fn = Path(l.strip()).name
        # e.g. jhu_ont__12345_Myeloid.beta → key = jhu_ont__12345
        key = fn.replace(f'_{target}.beta', '')
        done_keys.add(key)
    done_by_target[target] = done_keys

MANIFEST_DF['done'] = MANIFEST_DF['sample_key'].apply(
    lambda k: is_done(k, done_by_target)
)

MANIFEST_DF.to_csv(LOCAL_CSV, index=False)
n_done = MANIFEST_DF['done'].sum()
print(f'  Saved → {LOCAL_CSV}  ({len(MANIFEST_DF)} rows)')
print(f'  Done: {n_done} / {len(MANIFEST_DF)}')
print(f'\n  Per source folder:')
print(MANIFEST_DF.groupby('source_folder')['done'].agg(['sum', 'count']).to_string())

## Cell 6 — Verify GCS Resource Cache

In [ ]:
def gcs_pkg_exists(gcs_prefix, pkg):
    r = subprocess.run(
        f"gsutil ls '{gcs_prefix}/{pkg}*' 2>/dev/null | wc -l",
        shell=True, capture_output=True, text=True
    )
    return int(r.stdout.strip() or 0) > 0

REQUIRED_WHEELS = ['pandas', 'numpy', 'scipy']
REQUIRED_DEBS   = ['libdeflate0', 'libhtscodecs2', 'libhts3', 'libncurses6', 'tabix']

print('=== Resource cache verification ===\n')
all_ok = True

def chk(label, cmd):
    global all_ok
    r  = subprocess.run(cmd, shell=True, capture_output=True)
    ok = r.returncode == 0
    all_ok = all_ok and ok
    print(f'  {"" if ok else ""} {label}')

chk('wgbstools tar.gz',    f'gsutil -q stat {WGBS_TOOLS_TAR}')
chk('hg38 CpG refs tar',   f'gsutil -q stat {HG38_REFS_TAR}')

print('  Python wheels:')
for pkg in REQUIRED_WHEELS:
    ok = gcs_pkg_exists(WHEELS_GCS, pkg)
    all_ok = all_ok and ok
    print(f'    {"" if ok else ""} {pkg}')

print('  Debian packages:')
for pkg in REQUIRED_DEBS:
    ok = gcs_pkg_exists(DEBS_GCS, pkg)
    all_ok = all_ok and ok
    print(f'    {"" if ok else ""} {pkg}')

print('  Marker BEDs:')
for target, gcs_path in TARGET_BEDS.items():
    ok = subprocess.run(f'gsutil -q stat {gcs_path}', shell=True, capture_output=True).returncode == 0
    all_ok = all_ok and ok
    print(f'    {"" if ok else ""} {target}')

print(f'\n{"ALL CLEAR" if all_ok else "MISSING RESOURCES — fix before submitting"}')

## Cell 7 — Write Python Worker Script

This Python script runs inside each dsub VM and performs the actual LLR sorting.  
It is uploaded to GCS and pulled by the bash job script at runtime.

In [ ]:
worker_script = r'''#!/usr/bin/env python3
"""
Cell type sorting worker — runs on dsub VM.
All paths injected via environment variables from the bash wrapper.
"""
import os, re, math, gzip, subprocess, sys
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd

SAMPLE_ID        = os.environ["SAMPLE_ID"]
SOURCE_FOLDER    = os.environ["SOURCE_FOLDER"]
PAT_GCS          = os.environ["PAT_GCS"]
OUT_ROOT_GCS     = os.environ["OUT_ROOT_GCS"]
WORKSPACE_BUCKET = os.environ["WORKSPACE_BUCKET"]
WORK_DIR         = Path(os.environ["WORK_DIR"])
TOOL_BIN         = Path(os.environ["TOOL_BIN"])
SAMPLE_KEY       = f"{SOURCE_FOLDER}__{SAMPLE_ID}"

print(f"SAMPLE_KEY   : {SAMPLE_KEY}")
print(f"PAT_GCS      : {PAT_GCS}")
print(f"OUT_ROOT_GCS : {OUT_ROOT_GCS}")

TARGET_BEDS_GCS = {
    "Myeloid":     f"{WORKSPACE_BUCKET}/results/markers/markers_S2/coarse_Myeloid_vs_Lymphoid/Markers.Myeloid.bed",
    "Lymphoid":    f"{WORKSPACE_BUCKET}/results/markers/markers_S2/coarse_Lymphoid_vs_Myeloid/Markers.Lymphoid.bed",
    "T_Cell":      f"{WORKSPACE_BUCKET}/results/markers/markers_S2/one_vs_all/T_Cell_vs_all/Markers.T_Cell.bed",
    "B_Cell":      f"{WORKSPACE_BUCKET}/results/markers/markers_S2/one_vs_all/B_Cell_vs_all/Markers.B_Cell.bed",
    "NK_Cell":     f"{WORKSPACE_BUCKET}/results/markers/markers_S2/one_vs_all/NK_Cell_vs_all/Markers.NK_Cell.bed",
    "Monocyte":    f"{WORKSPACE_BUCKET}/results/markers/markers_S2/one_vs_all/Monocyte_vs_all/Markers.Monocyte.bed",
    "Granulocyte": f"{WORKSPACE_BUCKET}/results/markers/markers_S2/one_vs_all/Granulocyte_vs_all/Markers.Granulocyte.bed",
}

METH_CHARS   = {"C", "1"}
UNMETH_CHARS = {"T", "0"}
TAU     = 1.053    # LLR threshold (log of 2.87)
MIN_HITS = 5       # Minimum informative CpGs per read
EPS      = 1e-4

def run(cmd, **kw):
    subprocess.run(cmd, check=True, **kw)

def infer_pat_base(pat_gz, n=20000):
    mn = math.inf
    with gzip.open(pat_gz, "rt") as f:
        for i, line in enumerate(f):
            if i >= n: break
            parts = line.rstrip("\n").split("\t")
            if len(parts) >= 2:
                try: mn = min(mn, int(parts[1]))
                except: pass
    return 1 if mn >= 1 else 0

def load_cpg_sizes(chrom_size_path):
    chrom_counts, chrom_offsets = {}, {}
    offset = 0
    with open(chrom_size_path) as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) >= 2:
                chrom, n = parts[0], int(parts[1])
                chrom_counts[chrom]  = n
                chrom_offsets[chrom] = offset
                offset += n
    return chrom_counts, chrom_offsets

def load_marker_params(bed_path, chrom_counts, chrom_offsets):
    params = {}  # { (j0, j1): (p_tg, p_bg) }
    with open(bed_path) as f:
        for line in f:
            if line.startswith("#"): continue
            cols = line.rstrip("\n").split("\t")
            if len(cols) < 11: continue
            chrom = cols[0]
            if chrom not in chrom_offsets: continue
            j0  = chrom_offsets[chrom] + int(cols[3])
            j1  = chrom_offsets[chrom] + int(cols[4])
            p_tg = max(EPS, min(1-EPS, float(cols[9])))
            p_bg = max(EPS, min(1-EPS, float(cols[10])))
            params[(j0, j1)] = (p_tg, p_bg)
    return params

def llr_score(pattern, start_j, p_tg_map, p_bg_map):
    llr, hits = 0.0, 0
    for k, c in enumerate(pattern):
        if c not in METH_CHARS and c not in UNMETH_CHARS:
            continue
        j = start_j + k
        # Find which marker interval this CpG falls in
        for (j0, j1), (p_tg, p_bg) in p_tg_map.items():
            if j0 <= j < j1:
                m = 1 if c in METH_CHARS else 0
                llr  += m * (math.log(p_tg) - math.log(p_bg)) + (1-m) * (math.log(1-p_tg) - math.log(1-p_bg))
                hits += 1
                break
    return llr, hits

# ── Main pipeline ─────────────────────────────────────────────────
print("\n[1/5] Loading CpG chromosome sizes...")
CPG_SIZE_PATH = WORK_DIR / "wgbs_tools" / "references" / "hg38" / "CpG.chrome.size"
chrom_counts, chrom_offsets = load_cpg_sizes(str(CPG_SIZE_PATH))

print("[2/5] Loading marker BED files...")
all_params = {}
for target, gcs_path in TARGET_BEDS_GCS.items():
    local_bed = WORK_DIR / f"Markers.{target}.bed"
    subprocess.run(["gsutil", "-q", "cp", gcs_path, str(local_bed)], check=True)
    all_params[target] = load_marker_params(str(local_bed), chrom_counts, chrom_offsets)
    print(f"  {target}: {len(all_params[target])} marker intervals")

print("[3/5] Downloading PAT file...")
local_pat = WORK_DIR / f"{SAMPLE_ID}.pat.gz"
subprocess.run(["gsutil", "-q", "cp", PAT_GCS, str(local_pat)], check=True)

base = infer_pat_base(str(local_pat))
print(f"  PAT base offset: {base}")

print("[4/5] Sorting reads...")
out_handles = {}
for target in TARGET_BEDS_GCS:
    out_path = WORK_DIR / f"{SAMPLE_KEY}_{target}.pat"
    out_handles[target] = open(str(out_path), "w")

counts = Counter()
LOG_THRESH = math.log(TAU)

with gzip.open(str(local_pat), "rt") as f:
    for line in f:
        cols = line.rstrip("\n").split("\t")
        if len(cols) < 4: continue
        chrom, start_1based, pattern, mult = cols[0], int(cols[1]), cols[2], cols[3]
        if chrom not in chrom_offsets: continue
        start_j = chrom_offsets[chrom] + (start_1based - base)

        best_target, best_llr = None, LOG_THRESH
        for target, params in all_params.items():
            llr, hits = llr_score(pattern, start_j, params, {})
            if hits >= MIN_HITS and llr > best_llr:
                best_target, best_llr = target, llr

        if best_target:
            out_handles[best_target].write(line)
            counts[best_target] += int(mult)
        counts["total"] += int(mult)

for fh in out_handles.values():
    fh.close()

print(f"  Total reads: {counts['total']:,}")
for target in TARGET_BEDS_GCS:
    pct = counts[target] / max(1, counts['total']) * 100
    print(f"  {target}: {counts[target]:,} ({pct:.1f}%)")

print("[5/5] bgzip + index + pat2beta + upload...")
for target in TARGET_BEDS_GCS:
    raw_pat = WORK_DIR / f"{SAMPLE_KEY}_{target}.pat"
    gz_pat  = WORK_DIR / f"{SAMPLE_KEY}_{target}.pat.gz"

    # bgzip
    subprocess.run(["bgzip", "-f", str(raw_pat)], check=True)
    # index
    subprocess.run([str(TOOL_BIN), "index", str(gz_pat), "-f"], check=True)
    # pat2beta
    beta = WORK_DIR / f"{SAMPLE_KEY}_{target}.beta"
    subprocess.run([str(TOOL_BIN), "pat2beta", str(gz_pat), "-f", "--genome", "hg38"], check=True)

    dest = f"{OUT_ROOT_GCS}/{target}/"
    for ext in [".pat.gz", ".pat.gz.tbi", ".beta"]:
        fp = WORK_DIR / f"{SAMPLE_KEY}_{target}{ext}"
        if fp.exists():
            subprocess.run(["gsutil", "-q", "cp", str(fp), dest], check=True)

    print(f"  {target}: uploaded")

print("\nDONE:", SAMPLE_KEY)
'''

with open(WORKER_PY_FILE, 'w') as fh:
    fh.write(worker_script)

# Upload worker to GCS
subprocess.run(f'gsutil -q cp {WORKER_PY_FILE} {WORKER_SCRIPT_GCS}', shell=True, check=True)
print(f'Worker script written: {WORKER_PY_FILE}')
print(f'Uploaded to          : {WORKER_SCRIPT_GCS}')

## Cell 8 — Write Bash Job Script

In [ ]:
job_script = f"""#!/bin/bash
set -euo pipefail

echo "========================================"
echo " SAMPLE    : ${{SAMPLE_ID}}"
echo " SOURCE    : ${{SOURCE_FOLDER}}"
echo " HOST      : $(hostname)"
echo " START     : $(date -u)"
echo "========================================"

WORK_DIR="/tmp/wgbs_celltype_${{SAMPLE_ID}}"
mkdir -p "$WORK_DIR"
cd "$WORK_DIR"
export WORK_DIR

# ── 0a. Python wheels (version-matched) ──────────────────────────
echo "[0a/4] Python wheels..."
mkdir -p wheels
for py_ver in 310 311 312 313; do
    gsutil -m -q cp "{WHEELS_GCS}/${{py_ver}}/*.whl" ./wheels/ 2>/dev/null || true
done
gsutil -m -q cp "{WHEELS_GCS}/*.whl" ./wheels/ 2>/dev/null || true

pip install --no-index --find-links ./wheels \\
    pandas numpy scipy \\
    --quiet --break-system-packages --root-user-action=ignore

# ── 0b. bgzip + tabix via .deb ────────────────────────────────────
echo "[0b/4] bgzip + tabix..."
mkdir -p debs
gsutil -m -q cp "{DEBS_GCS}/*.deb" ./debs/

install_deb() {{
    shopt -s nullglob
    local debs=( $1 )
    shopt -u nullglob
    [ ${{#debs[@]}} -eq 0 ] && return 0
    dpkg -i "${{debs[@]}}" 2>/dev/null || dpkg --force-depends -i "${{debs[@]}}" || true
}}

install_deb "./debs/libdeflate0*.deb"
install_deb "./debs/libhtscodecs2*.deb"
install_deb "./debs/libncurses6*.deb"
install_deb "./debs/libhts3*.deb"
install_deb "./debs/tabix*.deb"

command -v bgzip &>/dev/null && echo "  bgzip OK" || {{ echo "ERROR: bgzip missing"; exit 1; }}
command -v tabix &>/dev/null && echo "  tabix OK" || {{ echo "ERROR: tabix missing"; exit 1; }}

# ── 1. wgbstools + hg38 CpG refs ─────────────────────────────────
echo "[1/4] wgbstools + hg38 refs..."
gsutil -q cp {WGBS_TOOLS_TAR} .
tar -xzf wgbstools_package.tar.gz
TOOL_BIN="$WORK_DIR/wgbs_tools/wgbstools"
chmod +x "$TOOL_BIN"
export TOOL_BIN

gsutil -q cp {HG38_REFS_TAR} .
tar -xzf hg38_cpg_refs.tar.gz

# ── 2. Worker script ─────────────────────────────────────────────
echo "[2/4] Worker script..."
gsutil -q cp {WORKER_SCRIPT_GCS} celltype_worker.py

# ── 3. Run sorting worker ────────────────────────────────────────
echo "[3/4] Sorting..."
export SAMPLE_ID="${{SAMPLE_ID}}"
export SOURCE_FOLDER="${{SOURCE_FOLDER}}"
export PAT_GCS="${{PAT_GCS}}"
export OUT_ROOT_GCS="${{OUT_ROOT_GCS}}"
export WORKSPACE_BUCKET="{BUCKET}"
python3 celltype_worker.py

echo "========================================"
echo " DONE : ${{SAMPLE_ID}}"
echo " END  : $(date -u)"
echo "========================================"
"""

with open(SCRIPT_FILE, 'w') as fh:
    fh.write(job_script)
os.chmod(SCRIPT_FILE, 0o755)
print(f'Job script written: {SCRIPT_FILE}')

## Cell 9 — Preflight Checks

In [ ]:
print(f'Preflight checks [celltype | {MODE}]\n')
all_ok = True

def chk(label, cmd):
    global all_ok
    r  = subprocess.run(cmd, shell=True, capture_output=True)
    ok = r.returncode == 0
    all_ok = all_ok and ok
    print(f'  {"" if ok else ""} {label}')

chk('wgbstools tar.gz',    f'gsutil -q stat {WGBS_TOOLS_TAR}')
chk('hg38 CpG refs tar',   f'gsutil -q stat {HG38_REFS_TAR}')
chk('Worker script (GCS)', f'gsutil -q stat {WORKER_SCRIPT_GCS}')
chk('dsub config file',    f'test -f {CONFIG_FILE}')
chk('job script file',     f'test -f {SCRIPT_FILE}')

print('  Marker BEDs:')
for target, path in TARGET_BEDS.items():
    ok = subprocess.run(f'gsutil -q stat {path}', shell=True, capture_output=True).returncode == 0
    all_ok = all_ok and ok
    print(f'    {"" if ok else ""} {target}')

print(f'\n{"ALL CHECKS PASSED" if all_ok else "FIX ISSUES ABOVE"}')

## Cell 10 — Build Job TSV

In [ ]:
pending_df = MANIFEST_DF[~MANIFEST_DF['done']].copy()
if cfg['n_samples']:
    pending_df = pending_df.head(cfg['n_samples'])

queue = []
for _, row in pending_df.iterrows():
    queue.append({
        '--env SAMPLE_ID':     row['sample_id'],
        '--env SOURCE_FOLDER': row['source_folder'],
        '--env PAT_GCS':       row['pat_gcs'],
        '--env OUT_ROOT_GCS':  OUT_ROOT_GCS,
    })

if not queue:
    print('Queue empty — all samples already done!')
else:
    tsv_df = pd.DataFrame(queue)
    tsv_df.to_csv(TSV_FILE, sep='\t', index=False)

    rate     = 0.19 * (0.30 if cfg['preemptible'] else 1.0)
    est_cost = len(queue) * (25 / 60) * rate

    print(f'{"━"*55}')
    print(f'  MODE:          {MODE}')
    print(f'  To run:        {len(queue)}')
    print(f'  Skipped:       {MANIFEST_DF["done"].sum()} (already done)')
    print(f'  TSV:           {TSV_FILE}')
    print(f'  Disk/VM:       {cfg["disk_gb"]}GB')
    print(f'  ~min/sample:   25 min (all parallel)')
    print(f'  Est. cost:     ~${est_cost:.0f} total')
    print(f'{"━"*55}')
    print(f'\nFirst 3 rows:')
    print(tsv_df.head(3).to_string(index=False))

## Cell 11 — Submit Jobs

Set `CONFIRMED = True` to submit.

In [ ]:
CONFIRMED = False    # ← Set True to submit

_net = {}
with open(CONFIG_FILE) as f:
    for line in f:
        k, v = line.strip().split('=', 1)
        _net[k] = v
PET_SA         = _net['PET_SA']
NETWORK_ARG    = _net['NETWORK']
SUBNETWORK_ARG = _net['SUBNETWORK']

LOG_PATH = (
    f'{BUCKET}/logs/celltype_sorting/'
    f'{datetime.now().strftime("%Y%m%d_%H%M")}/'
)

n_jobs   = len(pd.read_csv(TSV_FILE, sep='\t')) if os.path.exists(TSV_FILE) else 0
rate     = 0.19 * (0.30 if cfg['preemptible'] else 1.0)
est_cost = n_jobs * (25 / 60) * rate

flags = [
    'dsub',
    '--provider google-batch',
    f'--project {PROJECT_ID}',
    f'--location {REGION}',
    f'--service-account {PET_SA}',
    '--use-private-address',
    f'--network {NETWORK_ARG}',
    f'--subnetwork {SUBNETWORK_ARG}',
    '--machine-type n2-standard-4',
    f'--boot-disk-size {cfg["disk_gb"]}',
    '--image google/cloud-sdk:latest',
    f'--script {SCRIPT_FILE}',
    f'--tasks {TSV_FILE}',
    f'--logging {LOG_PATH}',
    '--name wgbs-celltype-all7',
    '--timeout 2h',
]
if cfg['preemptible']: flags.append('--preemptible')
if cfg['wait']:        flags.append('--wait')

dsub_cmd = ' \\\n    '.join(flags)

print(f'{"━"*55}')
print(f'  MODE:        {MODE}')
print(f'  Jobs:        {n_jobs}')
print(f'  Machine:     n2-standard-4 (4 vCPU / 16GB RAM)')
print(f'  Disk:        {cfg["disk_gb"]}GB')
print(f'  Preemptible: {cfg["preemptible"]}')
print(f'  Timeout:     2h')
print(f'  Est. cost:   ~${est_cost:.0f}')
print(f'  Logs:        {LOG_PATH}')
print(f'{"━"*55}')
print(f'\nCommand:\n{dsub_cmd}\n')

if not CONFIRMED:
    print('LOCKED — set CONFIRMED = True to submit')
else:
    print(f'Submitting {n_jobs} jobs...')
    start  = time.time()
    result = subprocess.run(dsub_cmd, shell=True, capture_output=True, text=True)
    elapsed = (time.time() - start) / 60
    print(result.stdout)

    if result.returncode == 0:
        print(f'Submitted in {elapsed:.1f} min')
        print(f'All {n_jobs} jobs run simultaneously')
        print(f'Expected wall-clock: ~25 min total')
        print(f'Monitor with Cell 12')
    else:
        key = [l for l in result.stderr.split('\n')
               if any(kw in l for kw in ['Error', 'denied', 'permission', 'invalid'])]
        print(f'Failed (code {result.returncode})')
        print('\n'.join(key[-8:]))
        print(f'\nLogs: {LOG_PATH}')

## Cell 12 — Monitor Progress

In [ ]:
print(f'{"━"*55}')
print(f'  MONITORING — Cell Type Sorting (all 7 targets)')
print(f'  Output: {OUT_ROOT_GCS}/')
print(f'  Time:   {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print(f'{"━"*55}\n')

total = len(MANIFEST_DF)

current_done = {}
for target in TARGET_BEDS:
    r = subprocess.run(
        f'gsutil ls {OUT_ROOT_GCS}/{target}/*.pat.gz 2>/dev/null | wc -l',
        shell=True, capture_output=True, text=True
    )
    current_done[target] = int(r.stdout.strip() or 0)

n_done = min(current_done.values()) if current_done else 0
pct    = n_done / total * 100 if total else 0
bar    = '█' * int(50 * pct / 100) + '░' * (50 - int(50 * pct / 100))

print(f'  Overall (min across targets):')
print(f'  {n_done:>5} / {total}  [{bar}]  {pct:.1f}%\n')
print(f'  Per target:')
for target in TARGET_BEDS:
    n = current_done[target]
    p = n / total * 100 if total else 0
    b = '█' * int(20 * p / 100) + '░' * (20 - int(20 * p / 100))
    print(f'    [{b}] {n:>5}/{total:<5} ({p:5.1f}%)  {target}')

print(f'\n{"━"*55}')
if n_done == total:
    print('  ALL DONE!')
elif n_done == 0:
    print('  Jobs running or not yet submitted (expected ~25 min total)')
else:
    print(f'  {total - n_done} remaining. Re-run this cell to refresh.')

## Cell 13 — Read Failure Logs

In [ ]:
N_SAMPLES   = 3
LAST_N_LINES = 40

r = subprocess.run(
    f'gsutil ls {BUCKET}/logs/celltype_sorting/ 2>/dev/null | sort | tail -1',
    shell=True, capture_output=True, text=True
)
latest = r.stdout.strip()

if not latest:
    print(f'No logs found under {BUCKET}/logs/celltype_sorting/')
else:
    print(f'Latest log dir: {latest}\n')
    r2 = subprocess.run(f"gsutil ls -r '{latest}'", shell=True, capture_output=True, text=True)
    stderr_logs = [l.strip() for l in r2.stdout.splitlines()
                   if l.strip().endswith('.log') and 'stderr' in l]

    for log in stderr_logs[:N_SAMPLES]:
        print(f'\n{"="*60}')
        print(f'Log: {log}')
        print(f'{"="*60}')
        r3 = subprocess.run(f"gsutil cat '{log}' | tail -n {LAST_N_LINES}",
                            shell=True, capture_output=True, text=True)
        print(r3.stdout)